In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType, FloatType, DecimalType

from pyspark.sql.window import Window

import pyspark.sql.functions as F

#reading the csv file
schema = StructType([
  StructField("Invoice", StringType(), True),
  StructField("StockCode", StringType(), True),
  StructField("Description", StringType(), True),
  StructField("Quantity", IntegerType(), True),
  StructField("InvoiceDate", TimestampType(), True),
  StructField("UnitPrice", DecimalType(10,2), True),
  StructField("Customer_id", StringType(), True),
  StructField("Country", StringType(), True)
])

silver_df = spark.read.format("delta") \
                      .option("schema", schema) \
                      .table("retail_analytics.reatail_dev.bronze_retail_raw")

In [0]:
silver_df.filter((F.col("Quantity") < 0) & (F.col("Price") > 0)).count()

In [0]:
display(silver_df.limit(5))

In [0]:
silver_df.count()

In [0]:
#dropping the duplicates
silver_df = silver_df.dropDuplicates()

In [0]:
silver_df.count()

In [0]:
def quality_check(silver_df):
  """
  This function is used to check the quality of the data
  """
  silver_df = silver_df.withColumn(
      "record_type",
      F.when(F.col("Invoice").startswith("C"), "Cancelled")
      .when(
          (F.col("Quantity") < 0) & (F.col("Price") == 0),
          "Stock_Adjustment"
      )
      .when(
          (F.col("Quantity") < 0) & (F.col("Price") > 0),
          "Return"
      )
      .otherwise("Normal Record")
  )

  return silver_df


In [0]:
def products_mapping(silver_df):

    description_counts = (
        silver_df
        .filter(F.col("Description").isNotNull())
        .groupBy("StockCode", "Description")
        .agg(F.count("*").alias("description_count"))
        .orderBy("StockCode", F.desc("description_count"))
    )

    valid_descriptions = description_counts.filter(
    ~F.lower(F.col("Description")).isin(
        "missing",
        "damaged",
        "?",
        "wrong code"
    )
    )
    
    windows_spec = Window.partitionBy("StockCode").orderBy(F.desc("description_count"))

    canonical_products = (
        valid_descriptions.withColumn("rank", F.row_number().over(windows_spec))
        .filter(F.col("rank") == 1)
        .select("StockCode", "Description")
    )


    all_products = silver_df.select("StockCode").distinct()

    product_df = all_products.join(
        canonical_products,
        on="StockCode",
        how="left"
    )

    product_df = product_df.withColumn(
        "Description",
        F.coalesce(
            F.col("Description"),
            F.concat(F.lit("Unknown Product - "), F.col("StockCode"))
        )
    )

    product_mapping = product_df.select(
        "StockCode",
        F.col("Description").alias("canonical_description")
    )

    silver_df = silver_df.join(
        product_mapping,
        on="StockCode",
        how="left"
    )

    silver_df = (
        silver_df
        .drop("Description")
        .withColumnRenamed("canonical_description", "Description")
    )

    return silver_df



In [0]:
quality_check_df = quality_check(silver_df)

In [0]:
transformed_df = products_mapping(quality_check_df)

In [0]:
transformed_df.display()

In [0]:
transformed_df.write.format("delta") \
                    .mode("overwrite") \
                    .saveAsTable("retail_analytics.reatail_dev.silver_retail_clean")

In [0]:
%sql
SELECT COUNT(*) FROM retail_analytics.reatail_dev.silver_retail_clean